# Lab 2 — Clean, impute, visualise: `clients_raw.csv`
**Data Analysis — Master AI / Master RSD** · Chapter 2: Data exploration and preparation

*Clean, impute, visualise: the full cycle on a concrete case.*

**Lab objectives**
1. **Clean** a raw dataset: detect and fix duplicates, inconsistencies and format problems → *reliable data ready for analysis*.
2. **Handle missing values**: apply several imputation methods and compare their impact → *choose the most suitable method*.
3. **Visualise** the data with Matplotlib and Seaborn → *understand and communicate*.
4. **Document every decision** (choices, justifications, results) in this notebook → *traceability and reproducibility*.

| Duration | Organisation | Data | Deliverable |
|---|---|---|---|
| 2 sessions of 2 h (4 h total) | Pairs · Jupyter notebook | `clients_raw.csv` (fictional, provided) | Completed notebook + conclusions |

**Prerequisites:** exploration from chapter 1 (Lab 1) and the methods of sections 4 to 7 of chapter 2.

⭐ **Continuity:** this lab follows Lab 1 — same dataset (`clients.csv`), in a "raw" version with injected defects.

⚠️ **Order of operations (must be respected)**: ① Deduplicate → ② Harmonise → ③ Type → ④ Validate ranges. This order avoids introducing errors in the following steps.

**How to use**: complete the cells marked `# TODO`, then run the **✅ Self-check** cell that follows. Answer the questions in the **✍️** cells.

---
## Setup

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer

sns.set_theme(style="whitegrid")   # tip: more readable plots
pd.set_option("display.max_columns", 20)

---
## Step 1 — Cleaning with Pandas: duplicates and inconsistencies
*A complete, commented script — to reproduce cell by cell.*

**Good practice:** check `shape` before/after each step — the notebook should **tell the story** of the cleaning.

### 1.0 Load the raw file
The separator is `;` (as in Lab 1).

In [6]:
df = ...   # TODO: pd.read_csv(...)

print(df.shape)
df.head()

AttributeError: 'ellipsis' object has no attribute 'shape'

In [ ]:
df.dtypes   # look at the type of the age column: why is it not numeric?

✅ **Self-check**

In [ ]:
assert df.shape == (1012, 8), f"Expected (1012, 8), got {df.shape} -> check sep=';'"
print("✅ 1.0 correct")

### 1.1 Duplicates: count, then remove
Count the **exactly duplicated** rows with `duplicated()`, then remove them with `drop_duplicates(keep="first")`.

In [ ]:
n_duplicates = ...   # TODO
print("Duplicates found:", n_duplicates)

df = ...   # TODO: remove duplicates (keep the first occurrence)
print(df.shape)

✅ **Self-check**

In [ ]:
assert n_duplicates == 12, "12 duplicates expected"
assert df.shape == (1000, 8), "1000 rows expected after deduplication"
assert df["id"].is_unique, "every id must now be unique"
print("✅ 1.1 correct")

### 1.2 Harmonise categories
First display the counts of `segment`, then replace the inconsistent categories.

**Good practice:** `replace()` **before** any analysis of categories — otherwise "a" and "A" count as two segments.

In [ ]:
df["segment"].value_counts()

In [ ]:
df["segment"] = ...   # TODO: replace "a" with "A" and "b" with "B"

df["segment"].value_counts()

✅ **Self-check**

In [ ]:
assert set(df["segment"].unique()) == {"A", "B", "C"}, "only A, B, C should remain"
assert df["segment"].value_counts()["A"] == 412, "we get back Lab 1's 412 A clients"
print("✅ 1.2 correct")

### 1.3 Types and value ranges
1. First look at which `age` values are **not numeric** (cell provided).
2. Convert `age` to numeric with `pd.to_numeric(..., errors="coerce")`: unreadable values become `NaN`.
3. Keep only **realistic** ages (between 0 and 120) with `between`.

**Good practice:** `errors="coerce"` turns unreadable values into `NaN` — handled in the next step.

In [ ]:
age_num = pd.to_numeric(df["age"], errors="coerce")
print("Non-numeric values:", df.loc[age_num.isna(), "age"].tolist())
print("Out-of-range values:", df.loc[~age_num.between(0, 120) & age_num.notna(), "age"].tolist())

In [ ]:
df["age"] = ...   # TODO: numeric conversion
df = ...          # TODO: keep ages between 0 and 120

print(df.shape)

✅ **Self-check**

In [ ]:
assert df.shape == (988, 8), f"Expected (988, 8), got {df.shape}"
assert df["age"].dtype == "float64", "age must be numeric"
assert df["age"].between(18, 87).all(), "the remaining ages are those from Lab 1 (18 to 87)"
print("✅ 1.3 correct — Final result: (988, 8) ← 12 duplicates removed, age ranges validated")

✍️ **Question:** `between(0, 120)` also removes the ages that became `NaN`. Why is that acceptable here? When should we **impute** them instead?

…

### 1.4 Validate the income range
In Lab 1 we found a client at **999,999 DA**: it is a data-entry error code, not a real income.
Replace this value with `NaN` (it will be imputed in step 2).

In [ ]:
print("Clients at 999,999:", (df["income"] == 999999).sum())

# TODO: replace 999999 with NaN in the income column (hint: df.loc[condition, "income"] = np.nan)

print("Max income:", df["income"].max(), "| NaN:", df["income"].isna().sum())

✅ **Self-check**

In [ ]:
assert df["income"].max() < 200000, "the 999999 code must be gone"
assert df["income"].isna().sum() == 38, "37 original NaN + 1 error code = 38"
assert df.shape == (988, 8), "do not drop the row: only the income is wrong"
print("✅ 1.4 correct")

✍️ **Decision to justify:** why replace the income with `NaN` rather than **dropping the row**?

…

---
## Step 2 — Handling missing values: comparing three methods
*Same dataset, three imputation strategies, three results to discuss.*

1. **Deletion** — drop incomplete rows (`dropna`)
2. **Median** — impute with the median (robust to outliers)
3. **k-NN** — impute with the *k* nearest neighbours

### 2.0 Missing-value rate before treatment

In [ ]:
rate = ...   # TODO: share of NaN per column, rounded to 3 decimals
rate

✅ **Self-check**

In [ ]:
assert rate["income"] == 0.038, "≈ 3.8% of incomes missing"
assert rate.drop("income").sum() == 0, "only income has missing values"
print("✅ 2.0 correct")

### 2.1 Apply the three methods
scikit-learn imputers only work on **numeric** columns: we use `num_cols` (without `id`, which has no statistical meaning).

`fit_transform` returns a NumPy array: we put it back into a `DataFrame` with the **same columns and the same index** as `df`.

In [ ]:
num_cols = ["age", "income", "n_children", "tenure", "n_purchases"]

# Method 1: deletion
df_del = ...   # TODO

# Method 2: median (robust to outliers)
imp = ...      # TODO: SimpleImputer(...)
df_med = pd.DataFrame(..., columns=num_cols, index=df.index)   # TODO

# Method 3: k nearest neighbours
knn = ...      # TODO: KNNImputer(...) with k = 5
df_knn = pd.DataFrame(..., columns=num_cols, index=df.index)   # TODO

✅ **Self-check**

In [ ]:
assert len(df_del) == 950, "988 - 38 incomplete rows = 950"
assert df_med.shape == df_knn.shape == (988, 5)
assert df_med.isna().sum().sum() == 0 and df_knn.isna().sum().sum() == 0
assert (df_med.loc[df["income"].isna(), "income"] == df["income"].median()).all(), "median = a single value"
print("✅ 2.1 correct")

### 2.2 Compare the three strategies
Build a table giving, for each method: the number of **rows kept**, of **missing values** (income), the **mean income**, the income **standard deviation**, and the difference in mean income compared with the raw data (**Δ vs raw**).

In [ ]:
methods = {"Raw (reference)": df, "Deletion": df_del, "Median": df_med, "k-NN (k=5)": df_knn}

comparison = pd.DataFrame({
    "rows":        ...,   # TODO: {name: len(d) for name, d in methods.items()}
    "missing":     ...,   # TODO: NaN in income
    "mean_income": ...,   # TODO
    "std_income":  ...,   # TODO
}).round(0)
comparison["delta_vs_raw"] = ...   # TODO: mean_income - mean income of the raw data
comparison

✅ **Self-check**

In [ ]:
assert list(comparison["rows"]) == [988, 950, 988, 988]
assert list(comparison["missing"]) == [38, 0, 0, 0]
assert comparison.loc["Deletion", "delta_vs_raw"] == 0
assert comparison.loc["Median", "delta_vs_raw"] < comparison.loc["k-NN (k=5)", "delta_vs_raw"] < 0
assert comparison.loc["Median", "std_income"] < comparison.loc["k-NN (k=5)", "std_income"] < comparison.loc["Raw (reference)", "std_income"]
print("✅ 2.2 correct")

### 2.3 Where do the imputed values land?
Provided cell: we overlay the distribution of **observed** incomes and the 38 values **imputed** by each method.

In [ ]:
missing = df["income"].isna()

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(df["income"], bins=40, color="lightgray", label="observed", ax=ax)
ax.vlines(df_med.loc[missing, "income"], 0, 60, color="tab:orange", lw=3, label="imputed — median (38 times the same value)")
sns.rugplot(df_knn.loc[missing, "income"], color="tab:purple", height=0.15, lw=2, label="imputed — k-NN", ax=ax)
ax.set_title("Income: observed vs imputed values")
ax.set_xlabel("Income (DA)")
ax.legend()
plt.show()

✍️ **Questions**
1. Why does **deletion** give **exactly** the same mean income as the raw data? What did we still lose?
2. Why does **median** imputation **lower** both the mean **and** the standard deviation?
3. Which method do you keep for the rest of the lab? Justify.

1. …
2. …
3. …

### 2.4 Going further — really evaluating the imputation
Comparing means does not tell us whether the imputed values are **correct**. Classic method: **hide** values we know, impute them, then measure the error.

1. Hide 100 **known** incomes (random draw, seed 0).
2. Impute with the median, raw k-NN, and k-NN on **standardised** data (`StandardScaler`, section 6 of the course).
3. Compare the **mean absolute error** (MAE) between imputed and true values.

In [ ]:
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
hidden = rng.choice(df.index[df["income"].notna()], size=100, replace=False)
truth = df.loc[hidden, "income"]

test = df[num_cols].copy()
test.loc[hidden, "income"] = np.nan

def mae(filled):
    """Mean absolute error on the 100 hidden incomes."""
    return (filled.loc[hidden, "income"] - truth).abs().mean()

# TODO: impute `test` with SimpleImputer(median) and KNNImputer(5), compute mae(...) for each
# TODO: standardised k-NN: sc.fit_transform -> KNNImputer -> sc.inverse_transform

✍️ **Your analysis:** which method makes the smallest error? Why does standardisation change the k-NN result?

…

⭐ **Bonus:** add the `segment` (one-hot encoded with `pd.get_dummies`) to the standardised k-NN variables, and also try a **median per segment** (`groupby(...).transform("median")`). What do you conclude?

### 2.5 Final choice and saving
We keep **k-NN**: replace the income in `df` with the one from `df_knn`, then save `clients_clean.csv` (reused in step 3).

In [ ]:
df_clean = df.copy()
df_clean["income"] = ...   # TODO: k-NN imputed income, rounded to the nearest hundred (round(-2))

df_clean.to_csv("clients_clean.csv", sep=";", index=False, encoding="utf-8")
df_clean.shape

✅ **Self-check**

In [ ]:
assert df_clean.shape == (988, 8) and df_clean.isna().sum().sum() == 0
assert (df_clean["income"] % 100 == 0).all(), "incomes rounded to the nearest hundred"
print("✅ 2.5 correct — clients_clean.csv saved")

---
## Step 3 — Visualising with Matplotlib and Seaborn
*Three plots are enough for most of the diagnosis.*

| Plot | Purpose |
|---|---|
| 1. **Histogram** | shape of the distribution, skewness, outliers |
| 2. **Boxplot** | compare income across segments (median, IQR, outliers) |
| 3. **Heatmap** | linear correlations; \|r\| > 0.8 → possible redundancy |

On `df_clean`, draw the three plots **side by side** (`plt.subplots(1, 3, figsize=(18, 4))`), with titles and axis labels.

⚠️ `sns.distplot` is deprecated → use `sns.histplot` (or `ax.hist`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# 1. Histogram: shape and outliers
# TODO: axes[0].hist(...) of income, bins=30; title "Income distribution", axes "Income (DA)" / "Count"

# 2. Boxplot: comparison by segment
# TODO: sns.boxplot(data=..., x="segment", y="income", ax=axes[1], ...); title "Income by segment"

# 3. Correlation heatmap (numeric variables, without id)
# TODO: sns.heatmap(df_clean[num_cols].corr(), annot=True, cmap="Blues", fmt=".2f", ax=axes[2], ...)

plt.tight_layout()
plt.show()

✍️ **One finding per plot**
1. **Histogram**: shape of the distribution? skewness? extreme values?
2. **Boxplot**: which segment has the highest income? are the gaps clear?
3. **Heatmap**: what is the strongest correlation? Is there any redundancy (|r| > 0.8)?

1. …
2. …
3. …

---
## Step 4 — Document every decision
*Deliverable: completed notebook + conclusions. Traceability and reproducibility.*

✍️ **Decision log** — fill in the table (one row per decision).

| # | Problem found | Decision | Justification | Rows before → after |
|---|---|---|---|---|
| 1 | … | … | … | … → … |
| 2 | … | … | … | … → … |
| 3 | … | … | … | … → … |
| 4 | … | … | … | … → … |
| 5 | … | … | … | … → … |

✍️ **Conclusions** (5 to 10 lines): quality of the data received, imputation method chosen, main lessons from the plots, limits of your work.

…

---
## Summary

| Step | Command | What we learned |
|---|---|---|
| Duplicates | `duplicated()`, `drop_duplicates()` | count **before** removing |
| Categories | `replace()`, `value_counts()` | harmonise **before** analysing |
| Types | `pd.to_numeric(errors="coerce")` | unreadable becomes NaN, handled later |
| Ranges | `between()` | validate realistic values |
| Missing values | `dropna()`, `SimpleImputer`, `KNNImputer` | compare methods, not just means |
| Visualisation | `hist`, `sns.boxplot`, `sns.heatmap` | distribution, groups, correlations |

Order to respect: **Deduplicate → Harmonise → Type → Validate ranges → Impute → Visualise**.